# Task 4.4 — Propellant Mass Variation

**Classical Mechanics · Project I · EETAC, UPC**

Compares three propellant loads (10, 50, 90 kg) with fixed thrust T = 1500 N.
Answers: initial acceleration, burnout altitude, and maximum apogee.

In [ ]:
# ── Download simulation dependencies from GitHub ─────────────────────────────
import urllib.request, os
BASE = "https://raw.githubusercontent.com/gdlcjoel-lab/PROYECTOMECANICA/main/Definitivos/"
for f in ["rocket_simulation.py", "plot_utils.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + f, f)
        print(f"Downloaded {f}")

# Use non-interactive backend so plots render inline in Colab
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
print("Setup complete ✓")

In [ ]:
import numpy as np
from rocket_simulation import simulate_rocket, compute_metrics, print_metrics, T0, ISP, MP, G0

PROP_MASSES = [10.0, 50.0, 90.0]
LABELS      = ["m_prop = 10 kg", "m_prop = 50 kg", "m_prop = 90 kg"]
COLORS      = ["#5B9BD5", "#2A6496", "#0D2B55"]
LINESTYLES  = ["-", "--", ":"]
MDOT = T0 / (G0 * ISP)

print("=" * 60)
print("  TASK 4.4 — PROPELLANT MASS VARIATION")
print("=" * 60)

results, metrics_list = [], []
for mp in PROP_MASSES:
    m0 = MP + mp; tb = mp / MDOT; a0 = T0/m0 - G0
    print(f"\n>>> m_prop={mp:.0f} kg  m0={m0:.0f} kg  t_burn={tb:.2f}s  a0={a0:.2f}m/s²")
    t, r, v, drag, mass, accel = simulate_rocket(m_propellant=mp)
    m_obj = compute_metrics(t, r, v)
    results.append((t, r, v, drag, mass, accel))
    metrics_list.append(m_obj)
    print_metrics(m_obj, f"m_prop = {mp:.0f} kg")

print("\n1. HOW DOES ACCELERATION CHANGE?")
for mp, m_obj in zip(PROP_MASSES, metrics_list):
    m0 = MP+mp; a0 = T0/m0-G0; tb = mp/MDOT
    print(f"   {mp:.0f}kg  a0={a0:.2f}m/s²  t_burn={tb:.2f}s  z_max={m_obj['z_max']/1000:.2f}km")

print("\n2. BURNOUT ALTITUDES:")
for mp, res in zip(PROP_MASSES, results):
    t_arr, r_arr = res[0], res[1]
    tb = mp/MDOT; idx = int(np.argmin(np.abs(t_arr-tb)))
    print(f"   {mp:.0f}kg  t_burn={tb:.2f}s  z_burn={r_arr[idx,2]:.1f}m")

## Plots
Altitude, speed, mass and acceleration vs time. Dotted lines mark burnout.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Task 4.4 — Propellant Mass Variation", fontsize=14, fontweight="bold", color="#0D2B55")
fig.patch.set_facecolor("white")
axes = axes.flatten()

ls = ["-", "--", ":"]
for i, (res, label, color, linestyle) in enumerate(zip(results, LABELS, COLORS, ls)):
    t, r, v, drag, mass, accel = res
    speed = np.linalg.norm(v, axis=1)
    axes[0].plot(t, r[:,2]/1000, color=color, linestyle=linestyle, linewidth=2, label=label)
    axes[1].plot(t, speed,       color=color, linestyle=linestyle, linewidth=2, label=label)
    axes[2].plot(t, mass,        color=color, linestyle=linestyle, linewidth=2, label=label)
    axes[3].plot(t, accel,       color=color, linestyle=linestyle, linewidth=2, label=label)
    axes[0].axvline(PROP_MASSES[i]/MDOT, color=color, linestyle=":", linewidth=1, alpha=0.5)
    axes[3].axvline(PROP_MASSES[i]/MDOT, color=color, linestyle=":", linewidth=1, alpha=0.5)

for ax, (title, xl, yl) in zip(axes, [
    ("Altitude vs Time (dotted = burnout)", "Time [s]", "Altitude [km]"),
    ("Speed vs Time", "Time [s]", "Speed [m/s]"),
    ("Mass vs Time", "Time [s]", "Mass [kg]"),
    ("Acceleration vs Time (dotted = burnout)", "Time [s]", "Acceleration [m/s²]"),
]):
    ax.set_title(title, fontweight="bold", color="#0D2B55")
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig("task_4_4_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved as task_4_4_plots.png")